### Toy semantic search: embed 20 sentences, find the most similar to a query.

#### Embedding choice: TF-IDF (via sklearn) rather than a neural embedding model.
#### Why: it needs no model download / internet access, runs instantly, and is the classic pre-transformer way search engines did this — great for learning the mechanics before you swap in real embeddings later (see the "upgrade path" note at the bottom of this file).

In [27]:
from sentence_transformers import SentenceTransformer

In [28]:
# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5871.26it/s]


In [29]:
# The sentences to encode
sentences = [
    "The cat sat quietly on the warm windowsill.",
    "Dogs are loyal companions and love to play fetch.",
    "The stock market rallied after the interest rate cut.",
    "She booked a flight to Tokyo for the cherry blossom season.",
    "The chef added fresh basil to the tomato sauce.",
    "Quantum computers use qubits instead of classical bits.",
    "He fixed the leaking faucet in the kitchen last night.",
    "The soccer team won the championship in overtime.",
    "Rising inflation is squeezing household budgets this year.",
    "The novel's plot twist surprised every reader.",
    "Astronauts conducted a spacewalk to repair the solar panel.",
    "My grandmother bakes the best apple pie every autumn.",
    "The startup raised a new round of venture funding.",
    "A gentle rain fell over the quiet mountain village.",
    "The professor explained neural networks with a whiteboard diagram.",
    "The puppy chased its tail around the living room.",
    "Central banks are watching inflation data closely.",
    "The hikers reached the summit just before sunset.",
    "The orchestra performed a moving symphony at the concert hall.",
    "Researchers trained a language model on billions of words.",
]

In [30]:
# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)

(20, 384)


In [31]:
# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)

tensor([[ 1.0000e+00,  1.3361e-01,  4.6687e-02, -7.9827e-02,  5.2311e-02,
          1.6106e-02,  4.6901e-02,  1.0546e-01, -8.0409e-02,  7.0095e-02,
          7.8516e-02,  3.5459e-02, -7.0789e-02,  2.3777e-01, -4.6291e-02,
          2.1718e-01, -3.8249e-02, -8.2074e-02,  1.3186e-01,  5.7750e-02],
        [ 1.3361e-01,  1.0000e+00,  8.8965e-02,  5.7644e-02,  5.4482e-02,
          3.8120e-02,  2.2027e-02,  8.2627e-03, -7.5860e-02,  1.4625e-01,
         -8.7400e-02,  1.3571e-01,  5.3124e-03,  1.9508e-02,  4.5211e-02,
          2.5892e-01, -4.3055e-02,  6.7033e-02,  1.6269e-02,  8.4262e-02],
        [ 4.6687e-02,  8.8965e-02,  1.0000e+00, -2.1216e-02, -8.2693e-04,
          8.6460e-02,  3.2983e-02,  1.0130e-01,  1.5895e-01,  9.8583e-02,
          3.5875e-02, -3.7849e-02,  2.1684e-01,  9.1229e-02, -3.1329e-03,
          1.1712e-01,  2.7670e-01,  1.3945e-01,  1.6724e-02, -3.7203e-02],
        [-7.9827e-02,  5.7644e-02, -2.1216e-02,  1.0000e+00,  1.9094e-01,
         -1.5154e-02,  8.4012e-02, 

In [32]:
# query = "How do rising interest rates affect the economy?"

query = "How can i know that the interest rates are rising?"

In [33]:
# 1. Fit TF-IDF on the sentence collection, then embed the query with the
#    SAME vectorizer so it lands in the same vector space.
# Note: no stop-word list here — sklearn's default English list is quite
# aggressive (it drops words like "interest"), which would zero out overlap
# on a corpus this small. On a bigger corpus you'd normally keep it.
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
# 1. Fit TF-IDF on the sentence collection, then embed the query with the
#    SAME vectorizer so it lands in the same vector space.
# Note: no stop-word list here — sklearn's default English list is quite
# aggressive (it drops words like "interest"), which would zero out overlap
# on a corpus this small. On a bigger corpus you'd normally keep it.
vectorizer = TfidfVectorizer()
sentence_vectors = vectorizer.fit_transform(sentences)
query_vector = vectorizer.transform([query])
query_vector = vectorizer.transform([query])

In [34]:
from sklearn.metrics.pairwise import cosine_similarity

# 2. Compare the query against every sentence with cosine similarity.
similarities = cosine_similarity(query_vector, sentence_vectors)[0]

In [35]:
# 3. Rank and show the top results.
top_k = 5
ranked_idx = np.argsort(-similarities)[:top_k]

In [36]:
print(f"Query: {query!r}\n")
print(f"Top {top_k} most similar sentences:\n")
for rank, idx in enumerate(ranked_idx, start=1):
    print(f"{rank}. ({similarities[idx]:.3f}) {sentences[idx]}")

Query: 'How can i know that the interest rates are rising?'

Top 5 most similar sentences:

1. (0.274) The stock market rallied after the interest rate cut.
2. (0.210) Rising inflation is squeezing household budgets this year.
3. (0.177) Central banks are watching inflation data closely.
4. (0.157) Dogs are loyal companions and love to play fetch.
5. (0.067) The cat sat quietly on the warm windowsill.
